# Digital Twin Compressive Sensing - Complete Workflow

This notebook provides a comprehensive workflow for digital twin-based compressive sensing including:
1. **Dataset Generation** - Creating synthetic channel data using DeepVerse
2. **Model Training** - Training the ConvPrecoder_MISO neural network
3. **Inference** - Testing the trained model and extracting measurement vectors
4. **Visualization** - Plotting measurement vectors and radiation patterns

## Overview
The project implements a neural network-based approach for beam selection in massive MIMO systems using compressive sensing principles. The model learns to predict optimal beam indices from channel measurements.


In [ ]:
# Import all necessary libraries
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io import loadmat, savemat
from tqdm import tqdm
from collections import OrderedDict
from pprint import pprint
import sklearn.utils

# PyTorch imports
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torch.utils.tensorboard import SummaryWriter
from torchinfo import summary

# Check if CUDA is available
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Set random seeds for reproducibility
def setup_seed(seed=0):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)

setup_seed(0)


## 1. Configuration and Helper Functions

First, let's define the configuration class and helper functions needed throughout the workflow.


In [ ]:
class configurations(object):
    def __init__(self):
        # Dataset parameters
        self.real_data_root = "datasets/DT1"
        self.synth_data_root = "datasets/DT1"
        self.train_csv = "train_data_idx.csv"
        self.test_csv = "test_data_idx.csv"
        self.N_BS = 32  # Number of BS antennas
        self.N_MS = 1   # Number of MS antennas
        self.M_BS = 4   # Number of BS measurement vectors
        self.M_MS = 1   # Number of MS measurement vectors

        # Training parameters
        self.num_train_data = 10240
        self.batch_size = 32
        self.learning_rate = 1e-2
        self.num_epochs = 200
        self.gpu = 0
        
        # Training type and paths
        self.DT = True  # Use digital twin data
        self.finetune = None
        self.load_model_path = None
        self.store_model_path = None

# Create configuration instance
config = configurations()
print("Configuration loaded:")
print(f"- N_BS (BS antennas): {config.N_BS}")
print(f"- M_BS (measurement vectors): {config.M_BS}")
print(f"- Training data points: {config.num_train_data}")
print(f"- Batch size: {config.batch_size}")
print(f"- Learning rate: {config.learning_rate}")
print(f"- Epochs: {config.num_epochs}")


## 2. Dataset Generation

We'll use the DeepVerse framework to generate synthetic channel data. This section includes the UPA codebook generation and dataset creation functions.


In [ ]:
def UPA_codebook_generator_DFT(Mx, My, Mz, over_sampling_x=1, over_sampling_y=1, over_sampling_z=1, ant_spacing=0.5):
    """
    Generates a DFT-based codebook for a Uniform Planar Array (UPA).
    
    Args:
        Mx, My, Mz: Number of antenna elements on each axis
        over_sampling_x, over_sampling_y, over_sampling_z: Oversampling factors
        ant_spacing: Antenna spacing (default: 0.5 wavelengths)
    
    Returns:
        F_CB: The generated UPA codebook
        all_beams: Matrix of all beam indices for each axis
    """
    antx_index = np.arange(Mx)
    anty_index = np.arange(My)
    antz_index = np.arange(Mz)
    
    codebook_size_x = over_sampling_x * Mx
    codebook_size_y = over_sampling_y * My
    codebook_size_z = over_sampling_z * Mz
    
    # Generate DFT-based codebooks for each dimension
    theta_qx = np.arange(codebook_size_x) * (2 * np.pi / codebook_size_x)
    F_CBx = (1 / np.sqrt(Mx)) * np.exp(-1j * np.outer(antx_index, theta_qx))
    
    theta_qy = np.arange(codebook_size_y) * (2 * np.pi / codebook_size_y)
    F_CBy = (1 / np.sqrt(My)) * np.exp(-1j * np.outer(anty_index, theta_qy))
    
    theta_qz = np.arange(codebook_size_z) * (2 * np.pi / codebook_size_z)
    F_CBz = (1 / np.sqrt(Mz)) * np.exp(-1j * np.outer(antz_index, theta_qz))
    
    # Combine codebooks using Kronecker product
    F_CBxy = np.kron(F_CBy, F_CBx)
    F_CB = np.kron(F_CBz, F_CBxy)
    
    # Generate beam indices
    beams_x = np.arange(1, codebook_size_x + 1)
    beams_y = np.arange(1, codebook_size_y + 1)
    beams_z = np.arange(1, codebook_size_z + 1)
    
    Mxx_Ind = np.tile(beams_x, codebook_size_y * codebook_size_z)
    Myy_Ind = np.tile(np.tile(beams_y, codebook_size_x).reshape(-1, order='F'), codebook_size_z)
    Mzz_Ind = np.tile(beams_z, codebook_size_x * codebook_size_y)
    
    all_beams = np.column_stack((Mxx_Ind, Myy_Ind, Mzz_Ind))
    
    return F_CB, all_beams

# Generate codebook for the configuration
num_tx_ant = config.N_BS
F_CB, beam_indices = UPA_codebook_generator_DFT(num_tx_ant, 1, 1)
print(f"Generated codebook shape: {F_CB.shape}")
print(f"Beam indices shape: {beam_indices.shape}")


In [ ]:
def generate_synthetic_dataset(config, num_scenes=2411):
    """
    Generate synthetic dataset for training and testing.
    This is a simplified version of the DeepVerse dataset generation.
    """
    print(f"Generating synthetic dataset with {num_scenes} scenes...")
    
    # Create output directory
    output_dir = config.synth_data_root
    os.makedirs(output_dir, exist_ok=True)
    
    # Generate random channel data (simplified for demonstration)
    # In practice, this would come from the DeepVerse framework
    num_rx_ant = config.N_MS
    num_tx_ant = config.N_BS
    
    # Initialize arrays
    all_beam_idx = np.zeros((num_scenes, 1))
    all_channel = np.zeros((num_scenes, num_rx_ant, num_tx_ant), dtype=np.complex64)
    
    # Generate synthetic channels with beam selection
    for i in tqdm(range(num_scenes), desc="Generating channels"):
        # Generate random complex channel
        channel_real = np.random.randn(num_rx_ant, num_tx_ant)
        channel_imag = np.random.randn(num_rx_ant, num_tx_ant)
        channel = (channel_real + 1j * channel_imag) / np.sqrt(2)
        
        # Find best beam using the codebook
        gain = np.abs(channel @ np.conj(F_CB))  # [rx, codebook_size]
        beam_idx = np.argmax(gain, axis=1)
        
        all_beam_idx[i] = beam_idx[0]  # Take first (and only) RX antenna
        all_channel[i] = channel
    
    # Save dataset
    dataset_path = os.path.join(output_dir, "dataset.mat")
    savemat(dataset_path, {
        "all_beam_idx": all_beam_idx,
        "all_channel": all_channel
    })
    
    # Generate train/test split indices
    train_indices = np.random.choice(num_scenes, size=int(0.8 * num_scenes), replace=False)
    test_indices = np.setdiff1d(np.arange(num_scenes), train_indices)
    
    # Save CSV files
    pd.DataFrame({"data_idx": train_indices}).to_csv(
        os.path.join(output_dir, config.train_csv), index=False
    )
    pd.DataFrame({"data_idx": test_indices}).to_csv(
        os.path.join(output_dir, config.test_csv), index=False
    )
    
    print(f"Dataset saved to: {dataset_path}")
    print(f"Training samples: {len(train_indices)}")
    print(f"Test samples: {len(test_indices)}")
    print(f"Channel shape: {all_channel.shape}")
    print(f"Beam indices shape: {all_beam_idx.shape}")
    
    return all_channel, all_beam_idx, train_indices, test_indices

# Generate or load dataset
dataset_path = os.path.join(config.synth_data_root, "dataset.mat")
if not os.path.exists(dataset_path):
    print("Dataset not found. Generating new synthetic dataset...")
    channels, beam_indices, train_idx, test_idx = generate_synthetic_dataset(config)
else:
    print(f"Loading existing dataset from: {dataset_path}")
    data = loadmat(dataset_path)
    channels = data['all_channel']
    beam_indices = data['all_beam_idx']
    print(f"Loaded dataset with {channels.shape[0]} samples")


## 3. Data Loading and Preprocessing

Now we'll implement the DataFeed class for loading and preprocessing the channel data.


In [ ]:
def create_samples(data_root, csv_path, random_state, num_data_point, portion, select_data_idx):
    """Create samples from the dataset with optional shuffling and selection."""
    # Load channel data and beam indices
    channel = loadmat(os.path.join(data_root, "dataset.mat"))['all_channel']
    beam_idx = loadmat(os.path.join(data_root, "dataset.mat"))['all_beam_idx']
    
    # Load data indices
    if select_data_idx is None:
        data_idx = pd.read_csv(os.path.join(data_root, csv_path))["data_idx"].to_numpy()
    else:
        data_idx = select_data_idx
    
    channel = channel[data_idx, ...]
    beam_idx = beam_idx[data_idx, ...]
    
    # Shuffle data
    channel, beam_idx, data_idx = sklearn.utils.shuffle(channel, beam_idx, data_idx, random_state=random_state)
    
    if num_data_point:
        channel = channel[:num_data_point, ...]
        beam_idx = beam_idx[:num_data_point, ...]
        data_idx = data_idx[:num_data_point, ...]
    else:
        num_data = beam_idx.shape[0]
        p = int(num_data * portion)
        
        channel = channel[:p, ...]
        beam_idx = beam_idx[:p, ...]
        data_idx = data_idx[:p, ...]
        
    # Normalize channels
    channel /= np.linalg.norm(channel, ord='fro', axis=(-1, -2), keepdims=True)  
    
    return channel, beam_idx, data_idx


class DataFeed(Dataset):
    """PyTorch Dataset class for loading channel data."""
    
    def __init__(self, data_root, csv_path, random_state=0, num_data_point=None, portion=1.0, select_data_idx=None):
        self.data_root = data_root
        self.channel, self.label, self.data_idx = create_samples(
            data_root, csv_path, random_state, num_data_point, portion, select_data_idx
        )
        
    def __len__(self):
        return len(self.label)
    
    def __getitem__(self, idx):
        channel = self.channel[idx, ...]
        label = self.label[idx, 0]
        data_idx = self.data_idx[idx, ...]
        
        channel = torch.tensor(channel, requires_grad=False)
        label = torch.tensor(label, requires_grad=False)
        data_idx = torch.tensor(data_idx, requires_grad=False)
        
        return channel.cfloat(), label.long(), data_idx.long()


# Test data loading
try:
    # Create test data loaders
    train_feed = DataFeed(config.synth_data_root, config.train_csv, num_data_point=100)
    test_feed = DataFeed(config.synth_data_root, config.test_csv, num_data_point=50)
    
    train_loader = DataLoader(train_feed, batch_size=config.batch_size, shuffle=True)
    test_loader = DataLoader(test_feed, batch_size=32)
    
    # Test loading a batch
    channel_batch, label_batch, idx_batch = next(iter(train_loader))
    
    print(f"Data loading successful!")
    print(f"Training samples: {len(train_feed)}")
    print(f"Test samples: {len(test_feed)}")
    print(f"Channel batch shape: {channel_batch.shape}")
    print(f"Label batch shape: {label_batch.shape}")
    print(f"Channel dtype: {channel_batch.dtype}")
    print(f"Label dtype: {label_batch.dtype}")
    
except Exception as e:
    print(f"Data loading failed: {e}")
    print("Make sure the dataset exists or run the dataset generation cell above.")


## 4. Model Definition

Let's define the ConvPrecoder_MISO neural network model for beam selection.
